In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Wprowadzenie do prymitywów

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>
<span id="qpu-access-patterns"></span>
## Dlaczego Qiskit wprowadził prymitywy?
Podobnie jak we wczesnych dniach komputerów klasycznych, gdy programiści musieli bezpośrednio manipulować rejestrami procesora, pierwotny interfejs QPU po prostu zwracał surowe dane z elektroniki sterującej.
Nie stanowiło to większego problemu, gdy QPU znajdowały się w laboratoriach i były dostępne wyłącznie dla badaczy.
Uznając, że większość programistów nie powinna musieć znać szczegółów konwersji surowych danych na 0 i 1, Qiskit wprowadził `backend.run` — pierwszą abstrakcję umożliwiającą dostęp do QPU w chmurze. Pozwoliło to programistom pracować na znajomym formacie danych i skupić się na szerszym obrazie.

Gdy dostęp do QPU stał się powszechniejszy i zaczęto opracowywać coraz więcej algorytmów kwantowych,
znów pojawiła się potrzeba abstrakcji wyższego poziomu. W odpowiedzi Qiskit wprowadził
interfejs prymitywów, zoptymalizowany pod kątem dwóch kluczowych zadań w tworzeniu algorytmów kwantowych:
szacowania wartości oczekiwanych (`Estimator`) i próbkowania obwodów (`Sampler`). Celem jest ponownie
pomoc programistom w skupieniu się na innowacjach, a nie na konwersji danych. Interfejs prymitywów zastępuje interfejs `backend.run`, ponieważ Sampler zapewnia ten sam bezpośredni dostęp do sprzętu, który oferował `backend.run`.

## Czym jest prymityw?
Systemy obliczeniowe są zbudowane na wielu warstwach abstrakcji. Abstrakcje pozwalają skupić się na
poziomie szczegółowości istotnym dla wykonywanego zadania. Im bliżej sprzętu, tym niższy poziom
abstrakcji jest potrzebny (np. może być konieczne przenoszenie lub manipulowanie danymi na poziomie instrukcji procesora). Im bardziej złożone zadanie chcesz wykonać, tym wyższy poziom
abstrakcji będzie wymagany (np. możesz korzystać z biblioteki programistycznej do wykonywania
obliczeń algebraicznych).

W tym kontekście *prymityw* to najmniejsza instrukcja przetwarzania — najprostszy element składowy, z którego
można stworzyć coś użytecznego na danym poziomie abstrakcji.

Ostatni postęp w obliczeniach kwantowych zwiększył potrzebę pracy na wyższych poziomach abstrakcji.
W miarę jak dziedzina zmierza ku większym jednostkom przetwarzania kwantowego (QPU) i bardziej złożonym przepływom pracy, punkt ciężkości przesuwa się z interakcji z sygnałami poszczególnych
qubitów na traktowanie urządzeń kwantowych jako systemów wykonujących określone zadania.

Dwa najczęstsze zadania komputerów kwantowych to próbkowanie stanów kwantowych i obliczanie wartości oczekiwanych.
Te zadania stały się motywacją do zaprojektowania prymitywów Qiskit: **Estimator** i **Sampler**.

- Estimator oblicza wartości oczekiwane obserwowalnych względem stanów przygotowanych przez Circuit.
- Sampler próbkuje rejestr wyjściowy z wykonania Circuit.

Krótko mówiąc, model obliczeniowy wprowadzony przez prymitywy Qiskit przybliża programowanie kwantowe
do tego, czym jest dziś programowanie klasyczne — gdzie skupiamy się mniej na szczegółach sprzętowych, a bardziej na wynikach,
które chcemy osiągnąć.

## Definicja prymitywów i ich implementacje
Istnieją dwa rodzaje prymitywów Qiskit: klasy bazowe oraz ich implementacje. Prymitywy Estimator i Sampler są zdefiniowane przez klasy bazowe prymitywów o otwartym kodzie źródłowym, które znajdują się w Qiskit SDK (w module [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). Dostawcy (tacy jak Qiskit Runtime) mogą używać tych klas bazowych do tworzenia własnych implementacji Sampler i Estimator. Większość użytkowników korzysta z implementacji dostawców, a nie bezpośrednio z klas bazowych.

### Klasy bazowe
Klasy `Base` prymitywów to abstrakcyjne klasy definiujące wspólny interfejs do implementowania prymitywów. Wszystkie inne klasy w module [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) dziedziczą po tych klasach bazowych. Programiści powinni z nich korzystać, jeśli chcą tworzyć własny model wykonania oparty na prymitywach dla konkretnego dostawcy. Klasy te mogą być również przydatne dla osób, które chcą wykonywać wysoko spersonalizowane przetwarzanie i uważają, że istniejące implementacje prymitywów są dla nich zbyt proste. Zwykli użytkownicy nie będą bezpośrednio korzystać z klas bazowych.

[`BaseEstimatorV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV1) i [`BaseSamplerV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV1) — chociaż prymitywy V1 są nadal dostępne, niniejsze przewodniki skupiają się na prymitywach V2, ponieważ są najnowsze i powszechniej stosowane.

[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) i [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) — referencyjne prymitywy Qiskit są zgodne z tymi specyfikacjami interfejsu.

<span id="implementations"></span>
### Implementacje
Wszystkie prymitywy są tworzone na podstawie klas bazowych, dlatego mają tę samą ogólną strukturę i sposób użycia. Na przykład format danych wejściowych dla wszystkich prymitywów Estimator jest taki sam. Istnieją jednak różnice w implementacjach, które sprawiają, że każda z nich jest wyjątkowa.

Oto implementacje klas bazowych prymitywów:

- [Prymitywy Qiskit Runtime](/guides/qiskit-runtime-primitives), [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) i [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2), zapewniają bardziej zaawansowaną implementację (na przykład z uwzględnieniem mitygacji błędów) jako usługę w chmurze. Ta implementacja klas bazowych prymitywów służy do uzyskiwania dostępu do sprzętu IBM Quantum&reg;.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) i [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) — implementacje referencyjne prymitywów korzystające z symulatora wbudowanego w Qiskit. Są zbudowane przy użyciu modułu Qiskit [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information), generując wyniki na podstawie idealnych symulacji wektorów stanu. Dostęp do nich odbywa się przez Qiskit. Szczegóły użycia znajdziesz w sekcji [Dokładna symulacja z prymitywami Qiskit](/guides/simulate-with-qiskit-sdk-primitives).

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) i [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) — możesz użyć tych klas, aby „opakować" dowolny zasób obliczeniowy kwantowy w prymityw. Dzięki temu możesz pisać kod w stylu prymitywów dla dostawców, którzy nie mają jeszcze interfejsu opartego na prymitywach. Klas tych można używać tak samo jak zwykłego Sampler i Estimator, z tym że należy je inicjalizować z dodatkowym argumentem `backend` wskazującym, na którym komputerze kwantowym uruchomić obliczenia. Dostęp do nich odbywa się przez Qiskit. Więcej informacji znajdziesz w przewodniku [prymitywy backend](/guides/get-started-with-backend-primitives).

## Opcje
Możesz przekazywać opcje do prymitywów, aby dostosować je do swoich potrzeb. Chociaż interfejs metody `run()` prymitywów jest wspólny dla wszystkich implementacji, ich opcje już nie są. Sprawdź dokumentację API konkretnej implementacji prymitywu, aby dowiedzieć się, jakie opcje obsługuje.

Na przykład zapoznaj się z tematami [Opcje Estimator](/guides/estimator-options) i [Opcje Sampler](/guides/sampler-options), aby poznać opcje prymitywów Qiskit Runtime, lub zajrzyj do [dokumentacji API Qiskit Aer](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html), aby sprawdzić opcje prymitywów Qiskit Aer.

## Zalety prymitywów Qiskit
Dzięki prymitywom użytkownicy Qiskit mogą pisać kod kwantowy dla konkretnego QPU bez konieczności jawnego
zarządzania każdym szczegółem. Ponadto dodatkowa warstwa abstrakcji może ułatwić
dostęp do zaawansowanych możliwości sprzętowych danego dostawcy. Na przykład, korzystając z prymitywów Qiskit Runtime,
możesz skorzystać z najnowszych osiągnięć w zakresie mitygacji i tłumienia błędów, przełączając opcje takie jak [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) prymitywu, zamiast tworzyć własną implementację tych technik.

Dla dostawców sprzętu natywna implementacja prymitywów oznacza możliwość zapewnienia użytkownikom bardziej „gotowego do użycia"
sposobu dostępu do funkcji sprzętowych, takich jak zaawansowane techniki post-processingu. Dzięki temu użytkownicy mogą łatwiej korzystać z najlepszych możliwości Twojego sprzętu.

## Następne kroki
> **Tip:** - Zapoznaj się z [danymi wejściowymi i wyjściowymi prymitywów](/guides/primitive-input-output).
> - Zapoznaj się ze szczegółowymi [przykładami](/guides/simulate-with-qiskit-sdk-primitives).
> - Ćwicz z prymitywami, przerabiając [lekcję o funkcjach kosztu](/learning/courses/variational-algorithm-design/cost-functions) w IBM Quantum Learning.
> - Przejrzyj [Tworzenie dostawcy](/guides/create-a-provider), aby dowiedzieć się, jak zaimplementować własne prymitywy Sampler i Estimator.
> - Zobacz [dokumentację API](https://docs.quantum.ibm.com/api/qiskit/primitives).
> - Przeczytaj [Migracja do prymitywów V2](/guides/v2-primitives).
> - Dowiedz się więcej o [prymitywach Qiskit Runtime](/guides/qiskit-runtime-primitives), które służą do uruchamiania Circuit na QPU IBM.